# uri-pipeline

Builds legaldocml.ch's own URI machinery from two existing, real, working pieces instead of
re-implementing either:

- **AKN side**: `lib/uris/frbr_uri` — resolves AKN FRBR URIs (Work/Expression/Manifestation) from
  jurisdiction profiles, and parses legacy source URLs (Fedlex, ZH-Lex) into structured fields.
- **ELI side**: the EU ELI Annotation Tool's `eli_annotation.form_configs.URIScheme` class
  (`spec/misc/sources/ELI/ELI/eli-annotation-tool/`) — the *exact* code the tool's own web UI uses to
  build ELI URIs from `{eli:property}` / `{eli:property|year}` templates filled with real ELI metadata
  field values, including SKOS `notation` resolution for controlled-vocabulary properties. See
  `eli-annotation-tool/USAGE-NOTES-FOR-LEGALDOCML.md` for the template syntax.

**Vocabulary definitions are hand-edited, not auto-generated** — the ELI Annotation Tool's Excel→SKOS
round-trip (`any2skos.py`) is used so vocabularies can be reviewed/edited as a spreadsheet before
(re-)import, per the user's explicit preference. See §2 below.

**Run with**: `pipeline/uri/.venv` (has `pandas`, `jupyter`, `pytest`, and the ELI Annotation Tool
installed editable from its `spec/misc/sources/ELI/ELI/eli-annotation-tool/` location).

```bash
cd pipeline/uri
python3 -m venv .venv   # already done
.venv/bin/pip install -e ../../spec/misc/sources/ELI/ELI/eli-annotation-tool pandas jupyter pytest
.venv/bin/jupyter lab uri.ipynb
```

**To use one of our own Excel files**: just point `VOCAB_XLSX` (next cell) at any `.xlsx` path — e.g.
`REPO_ROOT / "spec" / "ELI Mapping - WIP.xlsx"` (21 sheets, incl. "Schema.org Legislation (Mapping"
and the "ELI-DL URI components" sheet) or `REPO_ROOT / "spec" / "SLI.xlsx"` (has an `eli_components`
tab). `pd.read_excel(..., sheet_name=N)` (or a sheet name string) picks which sheet to load — omit
`sheet_name` to see the list of available sheets first via `pd.ExcelFile(VOCAB_XLSX).sheet_names`.

Two different things can happen next, depending on what the sheet actually contains:
- **Just reviewing/exploring** a sheet (e.g. the WIP mapping tables) — any structure is fine, this is
  just `pandas` reading a spreadsheet.
- **Converting to real SKOS RDF** via the next cell's `any2skos.xl2rdf(...)` — the sheet *must* follow
  the strict vocabulary format from `excel-vocab-specif.rst` (a header block, then a body starting at
  the row where column A says "Concept URI"). Most of our own WIP sheets aren't in this format yet —
  `any2skos` will just tell you plainly if a sheet doesn't qualify (see the "Feuil2" example earlier
  this session), it won't silently produce garbage.

## 0. Setup — import both toolkits

In [5]:
import sys, os, json
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent  # pipeline/uri -> legaldocml.ch
sys.path.insert(0, str(REPO_ROOT / "lib" / "uris"))

import pandas as pd

# AKN side
from frbr_uri.resolver import FRBRResolver
from frbr_uri.reference import LegalReferenceBuilder
from frbr_uri.parsers.url_parser import parse_legal_url

# ELI side (the ELI Annotation Tool's own URI-scheme machinery)
from eli_annotation import form_configs, vocabs, form_values, any2eli, any2skos
from eli_annotation.form_configs import URIScheme

# Loads values for jurisdiction
resolver = FRBRResolver()
ref_builder = LegalReferenceBuilder(resolver)
print("Jurisdictions loaded:", [p["jurisdiction"] for p in resolver.list_jurisdictions()])

Jurisdictions loaded: ['ch', 'ch-be-351', 'ch-vd-5586', 'ch-zg', 'ch-zh', 'ch-zh-261']


## 1. Vocabulary definitions — manual editing supported

Vocabularies are SKOS `ConceptScheme`s, authored as **Excel workbooks** (see
`eli-annotation-tool/doc/user_manual/excel-vocab-specif.rst` for the exact column format —
header block, then a body starting at the row where column A says "Concept URI"). This cell:

1. loads an existing vocabulary Excel file as a plain `pandas` `DataFrame` so it can be reviewed/edited
   like any other spreadsheet (in Excel itself, or right here in the notebook),
2. re-exports it through the Annotation Tool's own converter (`any2skos.xl2skos`) once you're happy with
   the edits — the *same* code path the web UI's vocabulary-import form uses, so anything valid here is
   valid there.

Swap `VOCAB_XLSX` for a real Swiss-jurisdiction vocabulary once one exists; for now this loads the
verified-working test fixture (`tests/data/test00.xlsx`, a "weekdays" vocabulary) purely to prove the
round-trip.

**To use one of our own Excel files**: just point `VOCAB_XLSX` (next cell) at any `.xlsx` path — e.g.
`REPO_ROOT / "spec" / "ELI Mapping - WIP.xlsx"` (21 sheets, incl. "Schema.org Legislation (Mapping"
and the "ELI-DL URI components" sheet) or `REPO_ROOT / "spec" / "SLI.xlsx"` (has an `eli_components`
tab). `pd.read_excel(..., sheet_name=N)` (or a sheet name string) picks which sheet to load — omit
`sheet_name` to see the list of available sheets first via `pd.ExcelFile(VOCAB_XLSX).sheet_names`.

Two different things can happen next, depending on what the sheet actually contains:
- **Just reviewing/exploring** a sheet (e.g. the WIP mapping tables) — any structure is fine, this is
  just `pandas` reading a spreadsheet.
- **Converting to real SKOS RDF** via the next cell's `any2skos.xl2rdf(...)` — the sheet *must* follow
  the strict vocabulary format from `excel-vocab-specif.rst` (a header block, then a body starting at
  the row where column A says "Concept URI"). Most of our own WIP sheets aren't in this format yet —
  `any2skos` will just tell you plainly if a sheet doesn't qualify (see the "Feuil2" example earlier
  this session), it won't silently produce garbage.

In [2]:
ELI_TOOL = REPO_ROOT / "spec" / "misc" / "sources" / "ELI" / "ELI" / "eli-annotation-tool"

VOCAB_XLSX = ELI_TOOL / "tests" / "data" / "test00.xlsx"  # <- point this at a real vocab when ready

vocab_df = pd.read_excel(VOCAB_XLSX, sheet_name=0, header=None)
vocab_df.head(20)

,0,1,2,3,4,5,6
0,ConceptScheme URI,http://data.sparna.fr/vocabularies/days,NaN,NaN,NaN,NaN,NaN
1,dct:title@en,Weekdays,NaN,NaN,NaN,NaN,NaN
2,dct:description@en,The days of the week,NaN,NaN,NaN,NaN,NaN
3,dct:description@fr,Les jours de la semaine,NaN,NaN,NaN,NaN,NaN
4,prefix:euvoc,http://publications.europa.eu/ontology/euvoc#,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Concept URI,skos:prefLabel@en,skos:prefLabel@fr,skos:definition@en,"skos:narrower[separator=',']",skos:notation,euvoc:status
7,http://data.sparna.fr/vocabularies/days#monday,Monday,Lundi,The day where you get back at work,NaN,MON,http://publications.europa.eu/resource/authori...
8,http://data.sparna.fr/vocabularies/days#tuesday,Tuesday,Mardi,The most productive day of the week.,NaN,TUE,NaN
9,http://data.sparna.fr/vocabularies/days#wednesday,Wednesday,Mercredi,NaN,NaN,WED,NaN


In [3]:
# Edit vocab_df above (or the source .xlsx directly), save it, then re-run this cell to convert
# it to real SKOS RDF — exactly what the web UI's "Import vocabulary" button does.
OUTPUT_DIR = REPO_ROOT / "pipeline" / "uri" / "vocab_output"
OUTPUT_DIR.mkdir(exist_ok=True)

any2skos.xl2rdf(str(VOCAB_XLSX), str(OUTPUT_DIR / "vocab.rdf"))
list(OUTPUT_DIR.glob("vocab-*.rdf"))

[PosixPath('/Users/mgajdo/Development/infrastructure/web/legaldocml.ch/pipeline/uri/vocab_output/vocab-http__data.sparna.fr_vocabularies_days.rdf')]

## 2. Construct URIs from `eli:` metadata (the web UI's own mechanism)

`URIScheme` is the class the Annotation Tool's admin form-configuration page uses internally: give it
a `{eli:property}` / `{eli:property|year}` template and a filled-in form's values, and it produces the
real URI — chaining abstract-legal-resource → legal-resource → legal-expression → format exactly per
`configuration.rst`'s "Defining the URI schemes" section.

This example reuses the tool's own verified test fixtures (a real filled-in "act" notice) to prove the
whole chain end-to-end — swap in a real Swiss form config once the cantonal/municipal ELI-URI design
(the open question from `DECISIONS.md`) is settled.

In [4]:
vocab_index = vocabs.VocabularyIndex(str(ELI_TOOL / "tests" / "data" / "vocabs"))

with open(ELI_TOOL / "tests" / "data" / "act-01.json") as fp:
    raw_vals = form_values.ELIFormValues.load_from_json(json.load(fp))
with open(ELI_TOOL / "tests" / "data" / "forms" / "act.json") as fp:
    form_cfg = form_configs.ELIFormConfig.load_from_json(json.load(fp), raw_vals, vocab_index)

vals = form_cfg.read_form_values(raw_vals)
entities = vals.extract_eli_entities()
eli_uris = any2eli.build_uris_for_entities(entities, form_cfg, vals)

for entity_type, by_context in eli_uris.items():
    for (lang, fmt), uri in by_context.items():
        print(f"{entity_type:24s} lang={lang!s:6} fmt={fmt!s:6} -> {uri}")

eli:LegalResource        lang=None   fmt=None   -> http://example/2017/ABC/ACT
elix:AbstractLegalResource lang=None   fmt=None   -> http://example/2017/ABC
eli:LegalExpression      lang=http://publications.europa.eu/resource/authority/language/ENG fmt=None   -> http://example/2017/ABC/ACT/EN
eli:LegalExpression      lang=http://publications.europa.eu/resource/authority/language/FRA fmt=None   -> http://example/2017/ABC/ACT/FR
eli:Format               lang=http://publications.europa.eu/resource/authority/language/ENG fmt=https://www.iana.org/assignments/media-types/application/pdf -> http://example/2017/ABC/ACT/EN/PDF
eli:Format               lang=http://publications.europa.eu/resource/authority/language/ENG fmt=http://publications.europa.eu/resource/authority/product-form/PRINT -> http://example/2017/ABC/ACT/EN/PRINT
eli:Format               lang=http://publications.europa.eu/resource/authority/language/FRA fmt=https://www.iana.org/assignments/media-types/application/pdf -> http://exam

## 3. Legacy → citation → new AKN → new ELI (the 4-column comparison)

One row per real legacy source. Column 4 ("new ELI URI") is filled two different ways depending on
jurisdiction, per the `eli_subdivision_uri` finding in `DECISIONS.md`:

- **Federal (Fedlex) sources**: the legacy URI already *is* a real ELI URI — no translation needed,
  column 1 and column 4 coincide.
- **Cantonal/municipal (ZH-Lex etc.) sources**: no real ELI URI exists yet — this is the open design
  question. Left blank below until that's settled; do **not** fill it with the AKN-path-dressed-as-ELI
  placeholder (`eli_subdivision_uri`) that caused the original confusion.

In [6]:
LEGACY_SOURCES = [
    {"url": "https://www.fedlex.admin.ch/eli/cc/1999/404/de", "article": "47", "paragraph": "2", "litera": "a"},
    {"url": "http://www.zhlex.zh.ch/Erlass.html?Open&Ordnr=170.4,01.09.1998,01.01.1999,12", "article": "3", "paragraph": "1"},
]

rows = []
for src in LEGACY_SOURCES:
    ref = ref_builder.from_url(src["url"], article=src.get("article"), paragraph=src.get("paragraph"), litera=src.get("litera"))
    is_federal = ref.parsed_url.platform.startswith("fedlex")
    rows.append({
        "legacy_uri": ref.source_url,
        "citation": ref.human_citation("de"),
        "new_akn_uri": ref.fragment_uri,
        "new_eli_uri": ref.source_url if is_federal else None,  # cantonal: intentionally blank, see above
    })

pd.DataFrame(rows)

,legacy_uri,citation,new_akn_uri,new_eli_uri
0,https://www.fedlex.admin.ch/eli/cc/1999/404/de,AS CH/404 Art. 47 Abs. 2 lit. a,/akn/ch/lei/1999-01-01/404/deu@1999-01-01#art_...,https://www.fedlex.admin.ch/eli/cc/1999/404/de
1,http://www.zhlex.zh.ch/Erlass.html?Open&Ordnr=...,SR 170.4 Art. 3 Abs. 1,/akn/ch-zh/gesetz/1998-09-01/170.4/deu@1999-01...,NaN


## 4. Export to `lib/uris/`

Not yet built — depends on §3's cantonal ELI-URI design being settled first. Once it is, this cell should
write per-jurisdiction resolver config (mirroring `lib/uris/frbr_uri/profiles/*.json`) that a real ELI
resolver can consume the same way `FRBRResolver` consumes the AKN-side profiles.

In [ ]:
# TODO: once the cantonal ELI-URI scheme is designed, write it here as a URIScheme-compatible
# template + a jurisdiction profile JSON, matching lib/uris/frbr_uri/profiles/ch-zh.json's shape.


## 5. Authoritative URI-component definitions

Single source of truth: `pipeline/uri/uri_components.yaml` (hand-edited — see the user preference for
manual editing, especially of vocabulary/component definitions). This cell just loads and renders it;
edit the YAML file directly, then re-run this cell and the export cell below it, rather than editing
the generated tables or the exported appendix Markdown by hand.

In [8]:
import yaml

with open("uri_components.yaml") as fp:
    components = yaml.safe_load(fp)

eli_core_df = pd.DataFrame(components["eli_core"]["components"])
eli_core_df

,name,recommended_format,remarks,status,notes
0,jurisdiction,2-letter country code,"Extended here to Swiss cantons/communes, not j...",mapped,lib/uris/frbr_uri profiles use ch / ch-zh / ch...
1,agent,none — member states define their own,Codes for administrative hierarchical structur...,gap,Not yet distinguished from `jurisdiction` in o...
2,subagent,none — member states define their own,Finer administrative substructure than `agent`.,gap,NaN
3,year,4 digits,NaN,mapped,frbr_uri resolver templates use {date} as YYYY...
4,month,2 digits,NaN,gap,Can be used without year/day per the guide; no...
5,day,2 digits,NaN,gap,NaN
6,type,none — member states define their own,"Nature of the act (law, decree, draft bill, et...",mapped,"frbr_uri profiles' document_types (act, ordina..."
7,subtype,none — member states define their own,NaN,mapped,frbr_uri profiles' per-doc-type `subtype` (e.g...
8,natural_identifier,none — member states define their own,NaN,mapped,"The {number} field in frbr_uri (SR number, LS ..."
9,domain,none — member states define their own,Thematic classification.,gap,NaN


### Subdivisions (ELI §3.7.6 vs. our AKN implementation)

In [9]:
subdiv = components["subdivisions"]
print("ELI recommended pattern:", subdiv["eli_pattern"]["class"])
for k, v in subdiv["eli_pattern"].items():
    if k != "class":
        print(f"  {k}: {v}")
print()
print("Our AKN-side implementation:", subdiv["our_akn_implementation"]["location"],
      "-", subdiv["our_akn_implementation"]["status"])

ELI recommended pattern: eli:LegalResourceSubdivision
  type_property: eli:type_subdivision
  type_property_note: Needs its own controlled vocabulary of subdivision types — not yet defined for CH.
  number_property: eli:number (optional)
  parent_link: eli:is_part_of (links to the whole act and/or parent subdivision)
  extensibility_note: eli:LegalResourceSubdivision is a subclass of eli:LegalResource, so any other ELI metadata can further refine a subdivision (e.g. its own date of entry into force).


Our AKN-side implementation: lib/uris/frbr_uri/fragment.py - mapped


### ELI-DL (draft legislation) components — not implemented anywhere yet

In [ ]:
eli_dl_df = pd.DataFrame(components["eli_dl"]["components"])
eli_dl_df

### AKN-only concepts (no ELI equivalent) — **not yet researched, flagged not fabricated**

Status: `not_started`. Needs a real pass over AKN-NC's `<mod>`/`<textualMod>`/`<repeal>`/`<insertion>`
amendment machinery before this section means anything.

In [ ]:
print("status:", components["akn_only"]["status"])
print("components found so far:", components["akn_only"]["components"] or "(none yet)")

### Per-jurisdiction ELI URI schemes

Mirrors `lib/uris/frbr_uri/profiles/*.json`'s existing jurisdiction set. `ch` (federal) is real —
Fedlex already implements ELI. Every canton/commune below is `not_designed` — real design work, not a
gap-fill; edit `uri_components.yaml`'s `jurisdictions` section directly as each one gets designed.

In [ ]:
juris_rows = [
    {"jurisdiction": jid, **data}
    for jid, data in components["jurisdictions"].items()
    if isinstance(data, dict)
]
pd.DataFrame(juris_rows)

### Export to `spec/input/appendix/uri-components.md`

In [ ]:
def render_appendix_md(components: dict) -> str:
    lines = ["# URI Components — authoritative reference", "",
             "Generated from `pipeline/uri/uri_components.yaml` — do not edit this file by hand;",
             "edit the YAML and re-run `pipeline/uri/uri.ipynb` instead.", ""]

    lines += ["## ELI core components", "", components["eli_core"]["description"].strip(), "",
              "| Component | Recommended format | Status | Notes |", "|---|---|---|---|"]
    for c in components["eli_core"]["components"]:
        lines.append(f"| `{{{c['name']}}}` | {c['recommended_format']} | {c['status']} | {c.get('notes', c.get('remarks', ''))} |")
    lines.append("")

    lines += ["## Subdivisions", "", components["subdivisions"]["description"].strip(), ""]
    lines.append(f"- ELI class: `{components['subdivisions']['eli_pattern']['class']}`")
    lines.append(f"- Our AKN implementation: `{components['subdivisions']['our_akn_implementation']['location']}` ({components['subdivisions']['our_akn_implementation']['status']})")
    lines.append("")

    lines += ["## ELI-DL (draft legislation) — status: " + components["eli_dl"]["status"], "",
              "| Component | Remarks |", "|---|---|"]
    for c in components["eli_dl"]["components"]:
        lines.append(f"| `{c['name']}` | {c['remarks']} |")
    lines.append("")

    lines += ["## AKN-only concepts (no ELI equivalent) — status: " + components["akn_only"]["status"], ""]

    lines += ["## Per-jurisdiction ELI URI schemes", "",
              "| Jurisdiction | Level | Status | Scheme |", "|---|---|---|---|"]
    for jid, data in components["jurisdictions"].items():
        if not isinstance(data, dict):
            continue
        lines.append(f"| `{jid}` | {data['level']} | {data['status']} | {data.get('scheme') or '(not designed)'} |")
    lines.append("")

    return "\n".join(lines)


appendix_path = REPO_ROOT / "spec" / "input" / "appendix" / "uri-components.md"
appendix_path.write_text(render_appendix_md(components))
print("Wrote", appendix_path)